# 02 — Policy Iteration en GridWorld

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1PB2XFEsQV4GXcg1pad7r3ml1FIIhUbOr)


Policy Iteration separa explícitamente:

1. **Policy Evaluation**
2. **Policy Improvement**

Para una política fija $pi$:

$$
V^\pi(s)=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V^\pi(s')
$$

Aquí **no hay `max`**, porque la política ya eligió la acción.



## Puente teoría ↔ código

Policy Iteration separa dos ideas que en Value Iteration aparecían juntas:

### 1. Policy Evaluation

La política $\pi$ **ya escogió la acción**, por eso aquí no aparece `max`:

$$
V_{k+1}^{\pi}(s)
=
R(s)+
\gamma
\sum_{s'}
T(s,\pi(s),s')V_k^\pi(s')
$$

### 2. Policy Improvement

Una vez conocemos $V^\pi$, volvemos a comparar las acciones:

$$
\pi_{\text{new}}(s)
=
\arg\max_a
\sum_{s'}
T(s,a,s')V^\pi(s')
$$

En el código:

| Teoría | Código |
|---|---|
| $s$ | `state` |
| $\pi(s)$ | `policy[state]` |
| $a$ | `action` |
| $s'$ | `next_state` |
| $R(s)$ | `grid.get_reward(state)` |
| $\gamma$ | `grid.gamma` |
| $T(s,a,s')$ | `prob` |
| $V^\pi(s')$ | `V[next_state]` |
| Policy Evaluation | `policy_evaluation(...)` |
| Policy Improvement | `policy_improvement(...)` |

**Regla para no enredarse:**  
- Evaluar política → **seguir** la acción que ya dice $\pi$.  
- Mejorar política → **comparar** todas las acciones.


In [2]:
import numpy as np

class GridWorld:
    """
    GridWorld usando la notación de lecture8-mdp:
      state  = (row, col)
      action = (dr, dc)
      R(s)   = recompensa del estado actual
      T(s,a,s') = P(s' | s,a)
    """

    def __init__(self, height=3, width=4):
        self.height = height
        self.width = width

        # Estados especiales
        self.wall = (1, 1)
        self.terminal_states = {
            (0, 3): +1.0,
            (1, 3): -1.0,
        }

        # Parámetros del MDP
        self.living_reward = -0.04
        self.gamma = 1.0
        self.p_intended = 0.8
        self.p_perpendicular = 0.1

        # Right, Down, Left, Up
        self.actions = [
            (0, 1),
            (1, 0),
            (0, -1),
            (-1, 0),
        ]

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state != self.wall

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # El terminal es absorbente: una vez allí no hay
        # una nueva decisión que tomar.
        if self.is_terminal(state):
            return self.terminal_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        # Dinámica del MDP:
        # T(s,a,s') = P(s' | s,a).
        # Para una acción elegida, esta función enumera los
        # posibles estados siguientes y sus probabilidades.
        """
        Devuelve [(next_state, probability), ...]
        para T(s,a,s') = P(s' | s,a).
        """
        # El terminal es absorbente: una vez allí no hay
        # una nueva decisión que tomar.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Perpendiculares a (dr,dc)
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, self.p_intended),
            (perp1, self.p_perpendicular),
            (perp2, self.p_perpendicular),
        ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (
                state[0] + next_action[0],
                state[1] + next_action[1],
            )

            if not self.is_valid_state(next_state):
                next_state = state

            transitions.append((next_state, prob))

        return transitions

In [3]:
ARROWS = {
    (0, 1): "→",
    (1, 0): "↓",
    (0, -1): "←",
    (-1, 0): "↑",
}

def print_values(grid, V, fmt="{:+.3f}"):
    for row in range(grid.height):
        line = []
        for col in range(grid.width):
            s = (row, col)
            if not grid.is_valid_state(s):
                line.append("  WALL  ")
            else:
                line.append(fmt.format(V[s]))
        print(" | ".join(line))

def print_policy(grid, policy):
    for row in range(grid.height):
        line = []
        for col in range(grid.width):
            s = (row, col)
            if not grid.is_valid_state(s):
                line.append(" # ")
            elif grid.is_terminal(s):
                line.append(" + " if grid.get_reward(s) > 0 else " - ")
            else:
                line.append(f" {ARROWS[policy[s]]} ")
        print(" | ".join(line))

## 1. Policy Evaluation

Vamos a hacerlo iterativamente para que el código siga de cerca la ecuación de Bellman:

$$
V_{k+1}^{\pi}(s)=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

Paramos cuando:

$$
\max_s|V_{k+1}^\pi(s)-V_k^\pi(s)|<\theta
$$


In [4]:
def expected_next_value(grid, state, action, V):
    # Calcula el valor esperado del siguiente estado:
    #
    #     Σ_{s'} T(s,a,s') V(s')
    #
    # Esta es la parte estocástica de Bellman.
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def policy_evaluation(grid, policy, threshold=1e-6, max_iter=10_000):
    # POLICY EVALUATION:
    # calculamos V^pi(s) manteniendo la política fija.
    #
    # V_{k+1}^pi(s) = R(s)
    #                    + gamma Σ_{s'} T(s,pi(s),s') V_k^pi(s')
    #
    # IMPORTANTE: aquí NO usamos max.
    # La acción ya viene determinada por policy[state].
    # Inicializamos V_0^pi(s)=0.
    # Luego propagaremos las recompensas siguiendo la política.
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        # Actualización sincrónica:
        # construimos V_{k+1} usando solamente V_k.
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:

                # pi(s): la política fija la acción.
                # A diferencia de Value Iteration, NO buscamos
                # todavía la mejor acción.
                action = policy[state]


                # Bellman para una política fija:
                # R(s) + gamma * valor esperado al seguir pi(s).
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma
                    * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(
                biggest_change,
                abs(V_new[state] - V[state])
            )

        V = V_new

        # Si V^pi prácticamente no cambia, la evaluación
        # de esta política ha convergido.
        if biggest_change < threshold:
            break

    return V, iteration + 1

## 2. Policy Improvement

Ahora sí comparamos acciones:

$$
\pi_{\text{new}}(s)=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$


In [5]:
def policy_improvement(grid, V):
    # POLICY IMPROVEMENT:
    # ahora sí preguntamos si existe una acción mejor.
    #
    # pi_new(s) = argmax_a Σ_{s'} T(s,a,s') V^pi(s')
    new_policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue


        # Aquí aparece el equivalente al 'max' de Value Iteration.
        # max(..., key=...) devuelve la ACCIÓN que maximiza
        # el valor esperado; por eso implementa argmax.
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(
                grid, state, action, V
            )
        )

    return new_policy

## 3. Policy Iteration completa

Inicializamos una política arbitraria y repetimos:

```text
policy
  ↓
evaluate
  ↓
improve
  ↓
¿cambió?
  ├─ sí → repetir
  └─ no → terminar
```

In [6]:
def policy_iteration(grid, threshold=1e-6, max_iter=100):
    # POLICY ITERATION completa:
    #
    # política inicial
    #       ↓
    # Policy Evaluation  → calcula V^pi
    #       ↓
    # Policy Improvement → construye una política greedy
    #       ↓
    # si la política cambió, repetimos.
    # Política inicial: RIGHT en todos los estados no terminales.
    initial_action = (0, 1)
    policy = {
        state: initial_action
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        V, eval_iterations = policy_evaluation(
            grid, policy, threshold=threshold
        )

        new_policy = policy_improvement(grid, V)


        # Contamos en cuántos estados cambió la acción.
        # Si changed == 0, la política es estable.
        changed = sum(
            new_policy[state] != policy[state]
            for state in new_policy
        )

        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iterations,
            "changed_actions": changed,
        })

        policy = new_policy

        # Política estable:
        # mejorarla ya no produce ninguna acción diferente.
        # Hemos alcanzado la política óptima.
        if changed == 0:
            break

    V, _ = policy_evaluation(
        grid, policy, threshold=threshold
    )

    return policy, V, history

In [7]:
grid = GridWorld()

policy_star, V_star, history = policy_iteration(grid)

print("Historia:")
for h in history:
    print(h)

print("\nV^π*(s):")
print_values(grid, V_star)

print("\nπ*(s):")
print_policy(grid, policy_star)

Historia:
{'policy_iteration': 1, 'evaluation_sweeps': 119, 'changed_actions': 5}
{'policy_iteration': 2, 'evaluation_sweeps': 24, 'changed_actions': 2}
{'policy_iteration': 3, 'evaluation_sweeps': 28, 'changed_actions': 1}
{'policy_iteration': 4, 'evaluation_sweeps': 30, 'changed_actions': 0}

V^π*(s):
+0.812 | +0.868 | +0.918 | +1.000
+0.762 |   WALL   | +0.660 | -1.000
+0.705 | +0.655 | +0.611 | +0.388

π*(s):
 →  |  →  |  →  |  + 
 ↑  |  #  |  ↑  |  - 
 ↑  |  ←  |  ←  |  ← 


## 4. Value Iteration vs Policy Iteration

### Value Iteration

$$
V_{k+1}(s)
=
R(s)+\gamma
\max_a\sum_{s'}T(s,a,s')V_k(s')
$$

El `max` está en cada actualización.

### Policy Iteration

**Evaluar:**

$$
V^\pi(s)=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V^\pi(s')
$$

**Mejorar:**

$$
\pi_{\text{new}}(s)
=
\arg\max_a\sum_{s'}T(s,a,s')V^\pi(s')
$$

La separación entre estas dos operaciones es la idea central.



## La diferencia esencial con Value Iteration

### Value Iteration

En cada actualización hacemos directamente:

$$
\boxed{
V_{k+1}(s)=R(s)+\gamma
\max_a\sum_{s'}T(s,a,s')V_k(s')
}
$$

### Policy Iteration

Separamos el proceso:

**1. Evaluar la política actual**

$$
\boxed{
V^\pi(s)=R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V^\pi(s')
}
$$

**2. Preguntar si podemos mejorarla**

$$
\boxed{
\pi_{\text{new}}(s)=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
}
$$

La pregunta que debes hacerte al leer el código es:

> **¿Estoy evaluando una política que ya tengo, o estoy escogiendo una acción mejor?**

Eso te dice inmediatamente si debe aparecer o no un `max`.
